# Finite Differences and the Step Size

A finite difference replaces the limit in the definition of the derivative by a small step $h$. Too large an $h$ gives truncation error; too small an $h$ gives rounding error. This notebook finds the best step and shows why automatic differentiation avoids the trade-off.

In [ ]:
Pollis.packages("Plots", "DifferentiationInterface", "ForwardDiff", "FiniteDiff")

In [ ]:
using Plots
f(x) = exp(x) * sin(x)
df(x) = exp(x) * (sin(x) + cos(x))   # exact derivative
x0 = 1.0

## Three formulas

- Forward: $\frac{f(x+h) - f(x)}{h}$, error $O(h)$
- Central: $\frac{f(x+h) - f(x-h)}{2h}$, error $O(h^2)$
- Complex step: $\frac{\operatorname{Im} f(x + ih)}{h}$, error $O(h^2)$ and no subtraction

In [ ]:
hs = 10.0 .^ (-1:-0.25:-16)
fwd = [abs((f(x0 + h) - f(x0)) / h - df(x0)) for h in hs]
cen = [abs((f(x0 + h) - f(x0 - h)) / 2h - df(x0)) for h in hs]
cst = [abs(imag(f(x0 + im * h)) / h - df(x0)) for h in hs]
plot(hs, max.([fwd cen cst], 1e-17), xscale=:log10, yscale=:log10, m=:circle, lw=2,
     label=["forward" "central" "complex step"], xlabel="h", ylabel="absolute error", legend=:bottomleft)
vline!([sqrt(eps()), cbrt(eps())], ls=:dot, c=:gray, label="sqrt(eps), cbrt(eps)")

The forward difference is best near $h = \sqrt{\epsilon} \approx 10^{-8}$ with about 8 correct digits, and the central difference near $h = \epsilon^{1/3}$ with about 11. The complex step keeps full precision for any tiny $h$.

## Automatic differentiation has no step

ForwardDiff evaluates $f$ on a dual number $x_0 + \varepsilon$ and reads off the derivative exactly.

In [ ]:
using DifferentiationInterface
import ForwardDiff, FiniteDiff
(exact = df(x0), forwarddiff = derivative(f, AutoForwardDiff(), x0), finitediff = derivative(f, AutoFiniteDiff(), x0),
 error_ad = derivative(f, AutoForwardDiff(), x0) - df(x0), error_fd = derivative(f, AutoFiniteDiff(), x0) - df(x0))

## Exercises

1. Repeat with $f(x) = |x - 1|$ at $x_0 = 1$. What does each method return at the kink?
2. Add noise: $f(x) = \exp(x)\sin(x) + 10^{-10}\,\text{randn}()$. Which formulas survive?